The objective of this notebook is to verify whether the results of P8 (Environmental Exploration) remain valid after the CeDInt dataset correction.

The analysis focuses on the NW zone during 2021–2022 and covers P8.1–P8.7.

We recalculate the main results using the corrected dataset and compare them with the values reported in the original P8 notebook

In [1]:
from google.colab import files
import pandas as pd

# Upload the corrected dataset
uploaded = files.upload()

filename = next(iter(uploaded))

# Read the CSV
df = pd.read_csv(filename, low_memory=False)

# Convert timestamps
df["timestamp_local"] = pd.to_datetime(df["timestamp_local"])

print("Loaded file:", filename)
print("Total rows:", len(df))
print("Columns:", len(df.columns))

Saving cedint_f1_model_ready.csv to cedint_f1_model_ready.csv
Loaded file: cedint_f1_model_ready.csv
Total rows: 470680
Columns: 31


In [2]:
# Select NW zone, years 2021-2022
nw = df[
    (df["zone"] == "NW") &
    (df["timestamp_local"].dt.year.isin([2021, 2022])) &
    (df["era"] == "hourly") &
    (df["recommendation"] == "Use") &
    (df["load_type"].isin([
        "Lighting", "Plugs & Sockets", "HVAC"
    ]))
].copy()

# Convert Wh to kWh
nw["energy_kwh"] = pd.to_numeric(
    nw["energy_wh_model"], errors="coerce"
) / 1000

# Separate the three energy series
lighting = nw[nw["load_type"] == "Lighting"].copy()
plugs = nw[nw["load_type"] == "Plugs & Sockets"].copy()
hvac = nw[nw["load_type"] == "HVAC"].copy()

print("Rows by load type:")
print(nw["load_type"].value_counts())

print("\nNegative energy values:")
print((nw["energy_kwh"] < 0).sum())

print("\nMissing values:")
print(nw[["energy_kwh", "temp_c",
          "hum_pct", "lum_lux", "occupancy"]].isna().sum())

Rows by load type:
load_type
HVAC               17520
Lighting           17520
Plugs & Sockets    17520
Name: count, dtype: int64

Negative energy values:
0

Missing values:
energy_kwh    2308
temp_c        2364
hum_pct       2364
lum_lux       2364
occupancy     2364
dtype: int64


In [3]:
# Create the validation table
checks = []

def compare(section, metric, original, corrected, decimals=3):
    value = (
        round(float(corrected), decimals)
        if decimals is not None
        else int(corrected)
    )

    checks.append({
        "P8": section,
        "Metric": metric,
        "Original P8": original,
        "Corrected data": value,
        "Status": "PASS" if original == value else "DIFFERENT"
    })

# P8.1 - Luminosity vs Lighting
a = lighting[["lum_lux", "energy_kwh"]].apply(
    pd.to_numeric, errors="coerce"
).dropna()

compare("P8.1", "Luminosity correlation",
        -0.372, a["lum_lux"].corr(a["energy_kwh"]))

compare("P8.1", "Observations",
        17113, len(a), None)

# P8.2 and P8.3 - Occupancy vs Energy
for section, data, old_means, old_counts in [
    ("P8.2", lighting, [0.196, 0.171], [8951, 8162]),
    ("P8.3", plugs, [1.248, 1.308], [8951, 8164])
]:
    valid = data[
        data["occupancy"].isin([0, 1]) &
        data["energy_kwh"].notna()
    ]

    summary = valid.groupby("occupancy")[
        "energy_kwh"
    ].agg(["mean", "count"])

    for occ in [0, 1]:
        compare(section, f"Mean energy, occupancy={occ}",
                old_means[occ], summary.loc[float(occ), "mean"])

        compare(section, f"Observations, occupancy={occ}",
                old_counts[occ], summary.loc[float(occ), "count"],
                None)

display(pd.DataFrame(checks))

,P8,Metric,Original P8,Corrected data,Status
0,P8.1,Luminosity correlation,-0.372,-0.372,PASS
1,P8.1,Observations,17113.000,17113.000,PASS
2,P8.2,"Mean energy, occupancy=0",0.196,0.196,PASS
3,P8.2,"Observations, occupancy=0",8951.000,8951.000,PASS
4,P8.2,"Mean energy, occupancy=1",0.171,0.171,PASS
5,P8.2,"Observations, occupancy=1",8162.000,8162.000,PASS
6,P8.3,"Mean energy, occupancy=0",1.248,1.248,PASS
7,P8.3,"Observations, occupancy=0",8951.000,8951.000,PASS
8,P8.3,"Mean energy, occupancy=1",1.308,1.308,PASS
9,P8.3,"Observations, occupancy=1",8164.000,8164.000,PASS


In [4]:
# P8.4 and P8.5 - Environmental correlations
for section, column, old_corr in [
    ("P8.4", "temp_c", 0.174),
    ("P8.5", "hum_pct", 0.059)
]:
    sample = hvac[[column, "energy_kwh"]].apply(
        pd.to_numeric, errors="coerce"
    ).dropna()

    compare(section, f"{column} correlation",
            old_corr, sample[column].corr(sample["energy_kwh"]))

    compare(section, "Observations",
            15957, len(sample), None)

# P8.4 - Temperature interval 22-24 degrees
temp = hvac[["temp_c", "energy_kwh"]].apply(
    pd.to_numeric, errors="coerce"
).dropna()

interval = temp[
    (temp["temp_c"] >= 22) &
    (temp["temp_c"] < 24)
]

compare("P8.4", "HVAC mean at 22-24 C",
        0.139, interval["energy_kwh"].mean())

compare("P8.4", "Observations at 22-24 C",
        2198, len(interval), None)

# P8.5 - Humidity distribution
hum = hvac[["hum_pct", "energy_kwh"]].apply(
    pd.to_numeric, errors="coerce"
).dropna()

compare("P8.5", "Humidity 30% observations",
        8429, (hum["hum_pct"] == 30).sum(), None)

compare("P8.5", "Humidity 31% observations",
        5798, (hum["hum_pct"] == 31).sum(), None)

display(pd.DataFrame(checks).tail(8))

,P8,Metric,Original P8,Corrected data,Status
10,P8.4,temp_c correlation,0.174,0.174,PASS
11,P8.4,Observations,15957.000,15957.000,PASS
12,P8.5,hum_pct correlation,0.059,0.059,PASS
13,P8.5,Observations,15957.000,15957.000,PASS
14,P8.4,HVAC mean at 22-24 C,0.139,0.139,PASS
15,P8.4,Observations at 22-24 C,2198.000,2198.000,PASS
16,P8.5,Humidity 30% observations,8429.000,8429.000,PASS
17,P8.5,Humidity 31% observations,5798.000,5798.000,PASS


In [5]:
# P8.6 - Occupancy by hour
oc = lighting[
    lighting["occupancy"].isin([0, 1])
].copy()

oc["hour"] = oc["timestamp_local"].dt.hour

hourly = oc.groupby("hour")[
    "occupancy"
].agg(["mean", "count"])

compare("P8.6", "Peak occupancy hour",
        14, hourly["mean"].idxmax(), None)

compare("P8.6", "Occupancy at 14:00 (%)",
        67.0, hourly.loc[14, "mean"] * 100, 1)

compare("P8.6", "Occupancy at 02:00 (%)",
        29.5, hourly.loc[2, "mean"] * 100, 1)


# P8.7 - Lighting by occupancy and time of day
lo = lighting[
    lighting["occupancy"].isin([0, 1]) &
    lighting["energy_kwh"].notna()
].copy()

lo["hour"] = lo["timestamp_local"].dt.hour

for period, hours, expected in [
    ("Daytime", range(9, 18),
     [(0, 0.069, 2707), (1, 0.072, 3699)]),
    ("Nighttime", range(0, 6),
     [(0, 0.274, 2841), (1, 0.304, 1447)])
]:
    group = lo[lo["hour"].isin(hours)].groupby(
        "occupancy"
    )["energy_kwh"].agg(["mean", "count"])

    for occ, old_mean, old_count in expected:
        compare("P8.7", f"{period}, occupancy={occ}",
                old_mean, group.loc[float(occ), "mean"])

        compare("P8.7", f"{period}, count, occupancy={occ}",
                old_count, group.loc[float(occ), "count"],
                None)

display(pd.DataFrame(checks).tail(11))

,P8,Metric,Original P8,Corrected data,Status
18,P8.6,Peak occupancy hour,14.000,14.000,PASS
19,P8.6,Occupancy at 14:00 (%),67.000,67.000,PASS
20,P8.6,Occupancy at 02:00 (%),29.500,29.500,PASS
21,P8.7,"Daytime, occupancy=0",0.069,0.069,PASS
22,P8.7,"Daytime, count, occupancy=0",2707.000,2707.000,PASS
23,P8.7,"Daytime, occupancy=1",0.072,0.072,PASS
24,P8.7,"Daytime, count, occupancy=1",3699.000,3699.000,PASS
25,P8.7,"Nighttime, occupancy=0",0.274,0.274,PASS
26,P8.7,"Nighttime, count, occupancy=0",2841.000,2841.000,PASS
27,P8.7,"Nighttime, occupancy=1",0.304,0.304,PASS


In [6]:
# FINAL P8 DATA QUALITY VALIDATION

comparison = pd.DataFrame(checks)

print("P8 DATA QUALITY VALIDATION")
print("NW Zone - Years 2021-2022")
print("=" * 50)

# Display all validation results
display(comparison)

# Count successful checks
passed = (comparison["Status"] == "PASS").sum()
total = len(comparison)

print("=" * 50)
print(f"Validation checks passed: {passed}/{total}")

if passed == total:
    print("\nFINAL VERDICT: PASS")
    print("All checked P8 results match the original results.")
    print("No changes to the main P8 conclusions are required.")
else:
    print("\nFINAL VERDICT: REVIEW REQUIRED")
    print("Some results need further investigation.")
    display(comparison[comparison["Status"] != "PASS"])

P8 DATA QUALITY VALIDATION
NW Zone - Years 2021-2022


,P8,Metric,Original P8,Corrected data,Status
0,P8.1,Luminosity correlation,-0.372,-0.372,PASS
1,P8.1,Observations,17113.000,17113.000,PASS
2,P8.2,"Mean energy, occupancy=0",0.196,0.196,PASS
3,P8.2,"Observations, occupancy=0",8951.000,8951.000,PASS
4,P8.2,"Mean energy, occupancy=1",0.171,0.171,PASS
5,P8.2,"Observations, occupancy=1",8162.000,8162.000,PASS
6,P8.3,"Mean energy, occupancy=0",1.248,1.248,PASS
7,P8.3,"Observations, occupancy=0",8951.000,8951.000,PASS
8,P8.3,"Mean energy, occupancy=1",1.308,1.308,PASS
9,P8.3,"Observations, occupancy=1",8164.000,8164.000,PASS


Validation checks passed: 29/29

FINAL VERDICT: PASS
All checked P8 results match the original results.
No changes to the main P8 conclusions are required.


Final Conclusion – P8 Data Quality Validation
This notebook validates the results of P8 (Environmental Exploration) after the CeDInt dataset correction.

The analysis was repeated using the corrected dataset for the NW zone during 2021–2022.

We checked the main results of P8.1–P8.7, including:
- Luminosity and lighting energy consumption.
- Occupancy and energy consumption.
- Temperature and HVAC energy consumption.
- Humidity and HVAC energy consumption.
- Hourly occupancy patterns.
- Daytime and nighttime lighting consumption.
Results: All 29 validation checks passed (29/29 PASS).

The recalculated results match the values reported in the original P8 notebook at the same level of precision.

Conclusion: The main findings and conclusions of P8 remain valid after the dataset correction. No changes to the original P8 results are required.

This validation confirms that the reported results can be reproduced using the corrected dataset